# Phase 4e: Empirical Feature Scoring + Post-hoc Selection

**Idea.** Score *every* candidate missing feature once (expensive: one SAE pass), download the table, then choose `Tox_min` / `BalCor_min` / AUC thresholds **afterwards** — re-running only cheap CPU analysis.

**Score = empirical discriminativeness.** Run the SAE (same checkpoint as 4b/4d) over a **labeled** toxic/safe corpus and, per feature, compute the AUC of its activation `g_i(x)` against the toxic label (AUC>0.5 = fires more on toxic). We merge this with the existing LLM `ToxicityScore` / `BalancedCorrelationScore` so you can threshold on any combination later.

**No test leakage.** Scoring uses a held-out **ToxicChat-style validation** set (`valid_1500.tsv`), not the test set.

### Two phases
- **A. Score & download** (cells 1–12, needs GPU): produces `feature_scores_full.tsv` and backs it up to Drive.
- **B. Analyze & select** (cells 13+, CPU only, re-runnable): load the table, view distributions, set thresholds, export a selection.

# Phase A — Score & download
## 1. Mount Drive & GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


## 2. Install libraries (numpy pinned to avoid the pandas ABI break)

In [ ]:
import os
!pip uninstall -y pandas numpy >/dev/null 2>&1
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets \
    pandas==2.2.2 "numpy==1.26.4" scikit-learn scipy
print('Installed. If a numpy ABI error appears later: Runtime -> Restart session, then re-run.')


## 3. Hugging Face login

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))


## 4. Clone repo (local disk)

In [ ]:
%%bash
if [ ! -d "/content/FAC-Synthesis" ]; then
  git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis && echo cloned
else
  cd /content/FAC-Synthesis && git pull origin main && echo pulled
fi


## 5. Central config
Only **scoring** inputs/outputs live here. Selection thresholds are chosen later in Phase B.

In [ ]:
import os

REPO          = '/content/FAC-Synthesis'
INTERPRET_DIR = f'{REPO}/sae_feature_analysis/interpret_features'
SAE_REPO      = 'Zhongzhi1228/sae_llama_l16_h65536'
MODEL_KEY     = 'llama'

# Labeled corpus for scoring (TARGET distribution, NOT the test set)
LABELED_TSV = f'{REPO}/our_work/synthesis/synthesis_data/step5/final_validate_test_datasets/valid_1500.tsv'

# Candidate missing features + their LLM scores (FeatureID, ToxicityScore, BalancedCorrelationScore, ...)
SCORED_FEATURES_TSV = f'{REPO}/our_work/synthesis/synthesis_data/run2_filtering_before_step4/input/second_annotator_all_scored_32tokens.tsv'

COLLECT_THRESHOLD = 0.0   # capture every g_i > this (keep 0.0)

WORK_DIR      = '/content/phase4e'
SAE_OUT_DIR   = f'{WORK_DIR}/4e_sae_out'
LOCAL_LABELED = f'{WORK_DIR}/labeled_corpus.tsv'
TEXTSPANS_TSV = f'{SAE_OUT_DIR}/threshold_{COLLECT_THRESHOLD}/textspans_group0.tsv'

# Where the scored table is saved (and re-loaded by Phase B)
DRIVE_4E_DIR   = '/content/drive/MyDrive/fac_synthesis (1)/step_4/4e_feature_selection'
SCORES_OUT     = f'{WORK_DIR}/feature_scores_full.tsv'
DRIVE_SCORES   = f'{DRIVE_4E_DIR}/feature_scores_full.tsv'

os.makedirs(WORK_DIR, exist_ok=True)
for p in [LABELED_TSV, SCORED_FEATURES_TSV]:
    assert os.path.exists(p), f'Missing input: {p} (edit the path here)'
print('Config OK.\n  labeled :', LABELED_TSV, '\n  pool    :', SCORED_FEATURES_TSV)


## 6. Patch `generator.py` for 4-bit (same as 4b/4d)

In [ ]:
import os
path = f'{INTERPRET_DIR}/generator.py'
src = open(path).read()
src = src.replace('CACHE_DIR = "xxx/.cache/huggingface"',
                  'CACHE_DIR = os.environ.get("HF_CACHE_DIR", "/root/.cache/huggingface")')
old_load = '''        self._model = trf.AutoModelForCausalLM.from_pretrained(
            self._name,
            cache_dir=CACHE_DIR,
            torch_dtype=self._dtype,
            device_map=maps
        )'''
new_load = '''        from transformers import BitsAndBytesConfig
        _use_4bit = os.environ.get("FAC_USE_4BIT", "1") == "1"
        if _use_4bit and self._device != "cpu":
            _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=tc.float16,
                bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR, quantization_config=_bnb, device_map=maps)
        else:
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR, torch_dtype=self._dtype, device_map=maps)'''
if new_load.split(chr(10))[0] not in src:
    assert old_load in src, 'model-load block not found (generator.py changed?)'
    open(path, 'w').write(src.replace(old_load, new_load)); print('Patched generator.py.')
else:
    print('generator.py already patched.')


## 7. Download SAE checkpoint (same as 4b/4d)

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
import os, shutil
pth = [f for f in list_repo_files(SAE_REPO) if f.endswith('.pth')]
assert pth, 'No .pth in SAE repo!'
print('Downloading:', pth[0])
SAE_PATH = hf_hub_download(repo_id=SAE_REPO, filename=pth[0])
if not os.path.basename(SAE_PATH).lower().startswith(('topk_l','sae_l','ae_l','topk5_l','topk6_l','topk7_l')):
    tgt = '/content/topk_l16_h65536.pth'
    if not os.path.exists(tgt): shutil.copy(SAE_PATH, tgt)
    SAE_PATH = tgt
print('SAE_PATH:', SAE_PATH)


## 8. Stage labeled corpus & keep row-aligned labels
`collect_spans` emits `TextID = row index`, so labels are read in the same order.

In [ ]:
import os, pandas as pd
df = pd.read_csv(LABELED_TSV, sep='\t', header=None, names=['text', 'label'])
df['label'] = df['label'].astype(int)
df.to_csv(LOCAL_LABELED, sep='\t', header=False, index=False)
N = len(df); labels = df['label'].values
print(f'labeled queries: {N} | toxic: {int(labels.sum())} ({100*labels.mean():.1f}%)')
assert set(df['label'].unique()) == {0, 1}, 'Need both classes for AUC.'


## 9. Run the SAE (`collect_spans.py`) over the labeled corpus

In [ ]:
import os, subprocess
assert os.path.exists(os.path.join(INTERPRET_DIR, 'collect_spans.py')), 'collect_spans.py not found'
os.makedirs(SAE_OUT_DIR, exist_ok=True)
env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
cmd = ['python', 'collect_spans.py', '0', MODEL_KEY, '0', '1',
       '--data-path', LOCAL_LABELED, '--threshold', str(COLLECT_THRESHOLD),
       '--sae-path', SAE_PATH, '--out-dir', SAE_OUT_DIR]
print('Running:', ' '.join(cmd), '\n(cwd =', INTERPRET_DIR, ')\n')
proc = subprocess.Popen(cmd, cwd=INTERPRET_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='')
assert proc.wait() == 0, 'collect_spans failed'
assert os.path.exists(TEXTSPANS_TSV), 'no textspans file produced'
print('\nSAE scoring done ->', TEXTSPANS_TSV)


## 10. Per-feature AUC (activation vs toxic label)
Dense activation vector per feature (0 where it didn't fire), then `roc_auc_score(label, activation)`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

# Robust read: keep only the first 3 tab fields per line; ignore the Span column,
# whose raw query text (quotes/newlines/tabs) breaks pd.read_csv ('EOF inside string').
rows = []
with open(TEXTSPANS_TSV, encoding='utf-8') as f:
    f.readline()  # skip header
    for line in f:
        parts = line.split('\t')
        if len(parts) < 3:
            continue
        try:
            rows.append((int(parts[0]), int(parts[1]), float(parts[2])))
        except ValueError:
            continue  # continuation line from a span that contained a newline -> skip
spans = pd.DataFrame(rows, columns=['NeuronID', 'TextID', 'Score'])
assert spans['TextID'].max() < N, 'TextID out of range — row alignment broken'
print(f'SAE rows: {len(spans)} | features fired: {spans.NeuronID.nunique()} | max NeuronID: {spans.NeuronID.max()}')

y = labels
recs = []
for fid, g in spans.groupby('NeuronID'):
    act = np.zeros(N)
    gm = g.groupby('TextID')['Score'].max()   # one feature may have several spans per query -> max
    act[gm.index.values] = gm.values
    n_fired = int((act > 0).sum()); pos_fired = int(((act > 0) & (y == 1)).sum())
    try: auc = float(roc_auc_score(y, act))
    except Exception: auc = np.nan
    recs.append({'FeatureID': int(fid), 'auc': auc, 'n_fired': n_fired, 'pos_fired': pos_fired,
                 'frac_of_fires_toxic': pos_fired / max(n_fired, 1)})
feat = pd.DataFrame(recs)
print('AUC computed for', len(feat), 'features')


## 11. Build the FULL scored table (every candidate feature)
Left-join AUC onto the whole candidate pool + its LLM scores. Features that never fired on the labeled set get `auc = 0.5` (non-discriminative). This is the table you threshold later.

In [ ]:
import pandas as pd
sc = pd.read_csv(SCORED_FEATURES_TSV, sep='\t')
sc['FeatureID'] = sc['FeatureID'].astype(int)
for c in ['ToxicityScore', 'BalancedCorrelationScore']:
    sc[c] = pd.to_numeric(sc[c], errors='coerce')
keep_cols = [c for c in ['FeatureID', 'ToxicityScore', 'BalancedCorrelationScore',
                         'SpanCount', 'ToxicSpanCount', 'DominantCategory', 'Summary'] if c in sc.columns]

scores = sc[keep_cols].merge(feat, on='FeatureID', how='left')
scores['n_fired']             = scores['n_fired'].fillna(0).astype(int)
scores['pos_fired']           = scores['pos_fired'].fillna(0).astype(int)
scores['auc']                 = scores['auc'].fillna(0.5)
scores['frac_of_fires_toxic'] = scores['frac_of_fires_toxic'].fillna(0.0)
scores = scores.sort_values('auc', ascending=False).reset_index(drop=True)

never = int((scores['n_fired'] == 0).sum())
print(f'candidate features: {len(scores)} | never fired on labeled set: {never} ({100*never/len(scores):.1f}%)')
print(scores[['auc', 'n_fired', 'pos_fired', 'ToxicityScore', 'BalancedCorrelationScore']]
      .describe(percentiles=[.5, .9, .99]).round(3).to_string())


## 12. Download — save the scored table to Drive

In [ ]:
import os
os.makedirs(DRIVE_4E_DIR, exist_ok=True)
scores.to_csv(SCORES_OUT, sep='\t', index=False)
scores.to_csv(DRIVE_SCORES, sep='\t', index=False)
print('Saved scored table:')
print('  local :', SCORES_OUT)
print('  drive :', DRIVE_SCORES)
print('\nPhase A done. You can now run Phase B (CPU only) any time — even after a restart.')


# Phase B — Analyze & select (CPU only, re-runnable)
Pick `Tox_min` / `BalCor_min` / AUC thresholds **here**, after seeing the distributions. Re-run these cells as many times as you like; no GPU needed.

## 13. Load the scored table
Works after a runtime restart — reads the saved file (Drive first, else local).

In [ ]:
import pandas as pd, os
# If running Phase B fresh after a restart, set these and mount Drive:
# from google.colab import drive; drive.mount('/content/drive')
DRIVE_SCORES = globals().get('DRIVE_SCORES', '/content/drive/MyDrive/fac_synthesis (1)/step_4/4e_feature_selection/feature_scores_full.tsv')
SCORES_OUT   = globals().get('SCORES_OUT',   '/content/phase4e/feature_scores_full.tsv')
_src = DRIVE_SCORES if os.path.exists(DRIVE_SCORES) else SCORES_OUT
scores = pd.read_csv(_src, sep='\t')
print('loaded', len(scores), 'features from', _src)
scores.head()


## 14. Distributions — to help choose thresholds
AUC spread, a survival curve over AUC cutoffs, and **AUC × Tox** / **AUC × BalCor** matrices (the same matrix style you used before, now with the empirical AUC axis).

In [ ]:
import numpy as np, pandas as pd

print('=== empirical AUC distribution (candidate pool) ===')
print(scores['auc'].describe(percentiles=[.5, .75, .9, .95, .99]).round(3).to_string())
fired = scores[scores['n_fired'] > 0]
print(f"\nfired on labeled set: {len(fired)}/{len(scores)} | AUC>=0.60: {(scores.auc>=0.60).sum()} | "
      f"AUC>=0.65: {(scores.auc>=0.65).sum()} | AUC>=0.70: {(scores.auc>=0.70).sum()}")

print('\n=== survival vs AUC threshold ===')
for a in [0.55, 0.58, 0.60, 0.62, 0.65, 0.70, 0.75, 0.80]:
    print(f'  AUC >= {a:.2f} : {(scores.auc >= a).sum():5d} features')

def matrix(row_col, row_ths, col_col, col_ths):
    print(f'\n=== {row_col} (rows) x {col_col} (cols): #features surviving BOTH ===')
    hdr = '  '.join(f'>={c:>4}' for c in col_ths)
    print(f'{row_col[:10]:>12} | ' + hdr)
    for r in row_ths:
        counts = [((scores[row_col] >= r) & (scores[col_col] >= c)).sum() for c in col_ths]
        print(f'        >={r:>4} | ' + '  '.join(f'{n:5d}' for n in counts))

matrix('auc', [0.55, 0.60, 0.65, 0.70], 'ToxicityScore', [1, 5, 6, 7, 8, 9, 10])
matrix('auc', [0.55, 0.60, 0.65, 0.70], 'BalancedCorrelationScore', [2.0, 3.0, 4.0, 5.0, 6.0])


## 15. Select — set your thresholds here (re-run freely)
Combine the empirical AUC with the LLM gates however you like. Leave a knob at its default to ignore it.
`TOP_N` optionally caps the result to your synthesis budget (by AUC).

In [ ]:
import pandas as pd

# ── tune these AFTER looking at cell 14 ──────────────────────────
AUC_MIN    = 0.60   # empirical discriminativeness floor (main axis)
MIN_FIRED  = 3      # require the feature to actually fire on >= this many labeled queries
TOX_MIN    = 0      # LLM toxicity gate (0 = ignore)
BALCOR_MIN = 0.0    # LLM coherence gate (0 = ignore)
TOP_N      = 635    # cap by AUC to match budget (None = no cap)
# ─────────────────────────────────────────────────────────────────

sel = scores[(scores.auc >= AUC_MIN) &
             (scores.n_fired >= MIN_FIRED) &
             (scores.ToxicityScore.fillna(0) >= TOX_MIN) &
             (scores.BalancedCorrelationScore.fillna(0) >= BALCOR_MIN)].copy()
sel = sel.sort_values('auc', ascending=False)
if TOP_N is not None:
    sel = sel.head(TOP_N)

print(f'criteria: AUC>={AUC_MIN}, n_fired>={MIN_FIRED}, Tox>={TOX_MIN}, BalCor>={BALCOR_MIN}, TOP_N={TOP_N}')
print(f'-> selected {len(sel)} features | mean AUC {sel.auc.mean():.3f} | median n_fired {int(sel.n_fired.median())}')
print(sel[['FeatureID','auc','n_fired','pos_fired','ToxicityScore','BalancedCorrelationScore']].head(15).to_string(index=False))


## 16. (Optional) Compare with your current Tox×BalCor 635
Reproduce the old selection and measure overlap — tells you how much is *selection* vs *generation*.

In [ ]:
OLD_TOX_MIN, OLD_BALCOR_MIN = 8, 6.0
old = set(scores.loc[(scores.ToxicityScore >= OLD_TOX_MIN) &
                     (scores.BalancedCorrelationScore >= OLD_BALCOR_MIN), 'FeatureID'])
new = set(sel['FeatureID'])
ov = old & new
print(f'old (Tox>={OLD_TOX_MIN} & BalCor>={OLD_BALCOR_MIN}): {len(old)}')
print(f'new selection                                 : {len(new)}')
print(f'overlap                                       : {len(ov)} ({100*len(ov)/max(len(new),1):.1f}% of new)')
weak_old = (scores.loc[scores.FeatureID.isin(old), 'auc'] <= 0.55).mean()
print(f'fraction of OLD 635 with AUC<=0.55 (~non-discriminative): {100*weak_old:.1f}%')


## 17. Export the selection for the next synthesis round

In [ ]:
import os
DRIVE_4E_DIR = globals().get('DRIVE_4E_DIR', '/content/drive/MyDrive/fac_synthesis (1)/step_4/4e_feature_selection')
os.makedirs(DRIVE_4E_DIR, exist_ok=True)
out_local = '/content/phase4e/new_selected_features.tsv'
out_drive = f'{DRIVE_4E_DIR}/new_selected_features.tsv'
cols = [c for c in ['FeatureID','auc','n_fired','pos_fired','ToxicityScore','BalancedCorrelationScore','Summary'] if c in sel.columns]
sel[cols].to_csv(out_local, sep='\t', index=False)
sel[cols].to_csv(out_drive, sep='\t', index=False)
print('exported', len(sel), 'features ->', out_drive)
print('Feed its FeatureID column into Phase 4a/4c (replacing the Tox×BalCor 635).')
